# Load Data

In [4]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'numHelped',
    'phase',
    'hasAnswer',
    'userId',
    'userFeNumHelped',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/question_centered_model_7d.parquet',
                    columns=required_columns)

# Get Descriptives

In [5]:
# Define columns for descriptive statistics
descriptive_columns = [
    'numHelped',
    'hasAnswer',
    'numHelpProvidedAT',
    'numQuestionsAskedAT'
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
]

def calculate_descriptives(data, columns):
    """Calculate descriptive statistics (μ, σ, min, max) for specified columns"""
    stats_dict = {}

    for col in columns:
        if col in data.columns:
            # Handle non-numeric columns by skipping them or converting
            if data[col].dtype in ['object', 'category']:
                continue

            stats_dict[col] = {
                'μ': data[col].mean(),
                'σ': data[col].std(),
                'min': data[col].min(),
                'max': data[col].max()
            }

    return pd.DataFrame(stats_dict).T

# Calculate overall descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===")
overall_stats = calculate_descriptives(df, descriptive_columns)
print(overall_stats.round(4))

# Special handling for numHelped by phase
print("\n=== NUMHELPED BY PHASE ===")
if 'numHelped' in df.columns and 'phase' in df.columns:
    # Phase 1 statistics
    phase1_data = df[df['phase'] == 1]
    if len(phase1_data) > 0:
        phase1_stats = {
            'μ': phase1_data['numHelped'].mean(),
            'σ': phase1_data['numHelped'].std(),
            'min': phase1_data['numHelped'].min(),
            'max': phase1_data['numHelped'].max()
        }
        print(f"Phase 1:")
        for stat, value in phase1_stats.items():
            print(f"  {stat}: {value:.4f}")

    # Phase 2 statistics
    phase2_data = df[df['phase'] == 2]
    if len(phase2_data) > 0:
        phase2_stats = {
            'μ': phase2_data['numHelped'].mean(),
            'σ': phase2_data['numHelped'].std(),
            'min': phase2_data['numHelped'].min(),
            'max': phase2_data['numHelped'].max()
        }
        print(f"\nPhase 2:")
        for stat, value in phase2_stats.items():
            print(f"  {stat}: {value:.4f}")

=== DESCRIPTIVE STATISTICS ===
                              μ         σ  min         max
numHelped                0.2755    1.9638  0.0    238.0000
hasAnswer                0.8545    0.3526  0.0      1.0000
numHelpProvidedAT       17.5398  142.6489  0.0  35743.0000
logNumQuestionsAskedAT   1.9041    1.6429  0.0      8.0324

=== NUMHELPED BY PHASE ===
Phase 1:
  μ: 0.2580
  σ: 1.9452
  min: 0.0000
  max: 238.0000

Phase 2:
  μ: 0.2930
  σ: 1.9820
  min: 0.0000
  max: 203.0000


# 1. Main Reciprocity Effect

In [6]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

sys.modules['stargazer.translators.statsmodels'].pd = pd

# Non-FE Models
formula1 = "numHelped ~ C(phase)*C(hasAnswer)"
formula2 = "numHelped ~ C(phase)*C(hasAnswer) + logNumQuestionsAskedAT + logNumHelpProvidedAT"
formula3 = "numHelped ~ C(phase)*C(hasAnswer)*logNumHelpProvidedAT + logNumQuestionsAskedAT"

# FE Models
formula4 = "userFeNumHelped ~ C(phase)*C(hasAnswer)"
formula5 = "userFeNumHelped ~ C(phase)*C(hasAnswer) + userFeLogNumQuestionsAskedAT + userFeLogNumHelpProvidedAT"
formula6 = "userFeNumHelped ~ C(phase)*C(hasAnswer)*userFeLogNumHelpProvidedAT + userFeLogNumQuestionsAskedAT"

model_names = ["1", "2", "3", "4", "5", "6"]
formulas = [formula1, formula2, formula3, formula4, formula5, formula6]

def fit_model(formula, model_name, data):
    """Fit a single model, save LaTeX results, and clean memory"""
    # print(f"Fitting Model {model_name}...")
    print(f"Formula: {formula}")

    try:
        # Clear memory before fitting
        gc.collect()

        # Fit the model
        model = smf.ols(formula=formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['userId']}
        )

        # print(f"✓ Model {model_name} fitted successfully")

        # Create individual Stargazer table for this model
        stargazer = Stargazer([model])
        stargazer.title(f"Model {model_name}: Effect of Receiving Answers on Providing Help")
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(True)

        # Generate LaTeX code
        latex_output = stargazer.render_latex()
        print(latex_output)

        # Clear the model from memory
        del model
        gc.collect()

        return True

    except MemoryError as e:
        print(f"✗ Memory error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False
    except Exception as e:
        print(f"✗ Error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False

successful_models = []
failed_models = []

for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"Processing {i+1}/{len(formulas)}")
    fit_model(formula, name, df)

Processing 1/6
Formula: numHelped ~ C(phase)*C(hasAnswer)
\begin{table}[!htbp] \centering
  \caption{Model 1: Effect of Receiving Answers on Providing Help}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{1}{c}{\textit{Dependent variable: numHelped}} \
\cr \cline{2-2}
\\[-1.8ex] & (1) \\
\hline \\[-1.8ex]
 C(hasAnswer)[T.1] & 0.167$^{***}$ \\
& (0.003) \\
 C(phase)[T.2] & 0.023$^{***}$ \\
& (0.001) \\
 C(phase)[T.2]:C(hasAnswer)[T.1] & 0.014$^{***}$ \\
& (0.001) \\
 Intercept & 0.115$^{***}$ \\
& (0.001) \\
\hline \\[-1.8ex]
 Observations & 47406940 \\
 $R^2$ & 0.001 \\
 Adjusted $R^2$ & 0.001 \\
 Residual Std. Error & 1.963 \\
 F Statistic & 3892.492$^{***}$ \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
Processing 2/6
Formula: numHelped ~ C(phase)*C(hasAnswer) + logNumQuestionsAskedAT + logNumHelpProvidedAT
\begin{table}[!htbp] \centering
  \captio